# MCP Demo — Telecom Tools

This notebook demonstrates the **MCP (Model Context Protocol) server** built in this project, in two layers:

1. **Layer 1 — MCP only.** Connect to the running MCP server, list its tools, and call them directly. No LLM. This proves the MCP protocol works and the tools are callable.
2. **Layer 2 — MCP + LLM.** Plug Azure OpenAI in front of the same MCP server. Watch the model pick which tools to call to answer customer questions. This is the full "agent" experience.

## Prerequisites

Before running this notebook, make sure two services are running in another terminal:

```bash
make seed         # one-time, populates data/telecom.db
make telecom_api  # in terminal 1 — :8001
make mcp_telecom  # in terminal 2 — :8765
```

(You don't need the `chatbot` service running for this notebook — we ARE the chatbot here.)

For Layer 2, also fill in `.env` with `AZURE_OPENAI_*` credentials.

## Setup

In [19]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# Load .env from the project root (notebook lives in notebooks/)
load_dotenv(Path('..') / '.env')

MCP_URL = os.getenv('MCP_TELECOM_URL', 'http://localhost:8765/mcp')
print(f'MCP server URL: {MCP_URL}')

MCP server URL: http://localhost:8765/mcp


## Layer 1 — Talk to MCP directly (no LLM)

MCP is a JSON-RPC protocol. The two operations we care about are:
- `list_tools` — "what tools do you offer?"
- `call_tool(name, arguments)` — "please run this tool"

Both go over a single HTTP connection (Streamable HTTP transport). The helper functions below open a session, run one operation, and close. In production you'd reuse a session; here per-call sessions keep each cell self-contained.

In [20]:
async def list_mcp_tools(url: str = MCP_URL):
    """Connect to an MCP server, return its tool list."""
    async with streamablehttp_client(url) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            response = await session.list_tools()
            return response.tools


async def call_mcp_tool(name: str, arguments: dict, url: str = MCP_URL) -> dict:
    """Call a single MCP tool. Returns:
      - is_error: True if the tool errored
      - content:  text — always valid JSON ('[]' for empty list, '[a,b,...]' for multi-item)
      - parsed:   pre-parsed Python value (dict or list)

    FastMCP returns one TextContent block per item for list-typed tools, no blocks
    at all for empty lists, and one block for dict-typed tools. We normalize to
    always-parseable JSON so notebook code doesn't have to care.
    """
    async with streamablehttp_client(url) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(name, arguments)
            blocks = [getattr(b, 'text', '') for b in (result.content or [])]
            blocks = [t for t in blocks if t]
            if not blocks:
                content = '[]'                          # list tool returned no items
            elif len(blocks) == 1:
                content = blocks[0]                     # dict tool, OR single-item list
            else:
                content = '[' + ','.join(blocks) + ']'  # multi-item list
            try:
                parsed = json.loads(content)
            except json.JSONDecodeError:
                parsed = content
            return {
                'is_error': bool(result.isError),
                'content':  content,
                'parsed':   parsed,
            }

print('Helpers defined.')

Helpers defined.


### List the tools the server exposes

In [21]:
tools = await list_mcp_tools()
print(f'MCP server exposes {len(tools)} tools:\n')
for t in tools:
    first_line = (t.description or '').strip().splitlines()[0]
    print(f'  • {t.name:<26} — {first_line[:80]}')

MCP server exposes 14 tools:

  • get_customer_profile       — Get a customer's profile: name, phone, email, account type (prepaid/postpaid),
  • get_current_plan           — Get the customer's current active plan: plan name, monthly fee, data quota (GB),
  • list_available_plans       — List plans the customer could switch to. Optional category filter:
  • change_plan                — Switch a customer's plan. TWO-STEP: call first with confirm=False to get a
  • get_balance_and_usage      — Get prepaid balance (if prepaid) and current cycle usage:
  • get_recent_bills           — Get the customer's recent bills with bill_id, amount, issue date, due date,
  • pay_bill                   — Pay a specific bill. payment_method must be one of: 'card', 'upi',
  • recharge_prepaid           — Top up a prepaid customer's balance. payment_method one of: 'card', 'upi',
  • list_addons                — List available addons. Optional category: 'data', 'roaming', 'international',
  • purchase_addo

In [22]:
# Look at a single tool's input schema (this is what the LLM will see)
tool = next(t for t in tools if t.name == 'change_plan')
print('Name:', tool.name)
print('Description:')
print(tool.description)
print('\nInput schema:')
print(json.dumps(tool.inputSchema, indent=2))

Name: change_plan
Description:
Switch a customer's plan. TWO-STEP: call first with confirm=False to get a
        proration preview (price difference, days remaining); only call with
        confirm=True AFTER the user explicitly confirms. Mutating action.

Input schema:
{
  "properties": {
    "customer_id": {
      "title": "Customer Id",
      "type": "string"
    },
    "new_plan_id": {
      "title": "New Plan Id",
      "type": "string"
    },
    "confirm": {
      "default": false,
      "title": "Confirm",
      "type": "boolean"
    }
  },
  "required": [
    "customer_id",
    "new_plan_id"
  ],
  "title": "change_planArguments",
  "type": "object"
}


### Call a tool directly

Let's pull a customer's profile. No LLM involved — we're directly invoking the MCP tool the same way an LLM would, just with hard-coded arguments.

In [23]:
result = await call_mcp_tool('get_customer_profile', {'customer_id': 'CUST002'})

if result['is_error']:
    print('TOOL ERROR:')
    print(result['content'])
    print('\nIs the telecom_api service running on :8001? Try `make telecom_api`.')
else:
    print(json.dumps(result['parsed'], indent=2))

{
  "customer_id": "CUST002",
  "name": "Priya Iyer",
  "phone": "+919900000002",
  "email": "priya@example.com",
  "account_type": "prepaid",
  "status": "active",
  "prepaid_balance": 35.0,
  "area_code": "BLR-02",
  "created_at": "2026-05-06T10:17:32+00:00"
}


### Build a picture by chaining calls

An LLM agent will eventually do this kind of multi-tool composition automatically. Here we do it by hand to show what the data looks like.

In [24]:
cid = 'CUST002'

calls = {
    'profile': await call_mcp_tool('get_customer_profile', {'customer_id': cid}),
    'plan':    await call_mcp_tool('get_current_plan',     {'customer_id': cid}),
    'usage':   await call_mcp_tool('get_balance_and_usage',{'customer_id': cid}),
    'bills':   await call_mcp_tool('get_recent_bills',     {'customer_id': cid, 'limit': 3}),
}

errors = {k: v['content'] for k, v in calls.items() if v['is_error']}
if errors:
    print('Some calls failed. Make sure telecom_api is running on :8001.\n')
    for k, msg in errors.items():
        print(f'  {k}: {msg}')
else:
    p  = calls['profile']['parsed']
    pl = calls['plan']['parsed']
    u  = calls['usage']['parsed']
    bs = calls['bills']['parsed']     # list (empty for prepaid customers)
    print(f"Customer  : {p['name']} ({p['phone']}) — {p['account_type']}, {p['status']}")
    print(f"Plan      : {pl['name']} (₹{pl['monthly_fee']}/mo) — expires {pl['expiry_date'][:10]}")
    print(f"Data usage: {u['data']['used_gb']}/{u['data']['quota_gb']} GB ({u['data']['pct_used']}% used)")
    print(f"Renewal in: {u['days_to_renewal']} day(s)")
    if isinstance(bs, list):
        suffix = ' (prepaid — no bills)' if not bs else ''
        print(f"Bills     : {len(bs)} recent{suffix}")
    else:
        # Edge case: list-tool returned exactly 1 item, came back as a single dict.
        print('Bills     : 1 recent')

Customer  : Priya Iyer (+919900000002) — prepaid, active
Plan      : Smart 199 (₹199.0/mo) — expires 2026-05-09
Data usage: 41.4/45.0 GB (92.0% used)
Renewal in: 2 day(s)
Bills     : 0 recent (prepaid — no bills)


### A mutating tool — preview vs apply

Mutating tools (`change_plan`, `recharge_prepaid`, `purchase_addon`, `pay_bill`, `block_sim`) follow a two-step pattern: call once with `confirm=False` to get a preview, then again with `confirm=True` to apply. This is the same pattern a real agent uses to ask for user approval before acting.

In [25]:
# PREVIEW: ask what would happen if CUST002 switched to PRO_599
preview = await call_mcp_tool('change_plan', {
    'customer_id': 'CUST002',
    'new_plan_id': 'PRO_599',
    'confirm': False,
})
if preview['is_error']:
    print('TOOL ERROR:', preview['content'])
else:
    print('PREVIEW:')
    print(json.dumps(preview['parsed'], indent=2))

# (we won't apply it — that would mutate the seed DB)

PREVIEW:
{
  "requires_confirmation": true,
  "summary": "Switch from 'Smart 199' (\u20b9199.0/mo) to 'Pro 599' (\u20b9599.0/mo). Estimated proration: \u20b926.67 for 2 day(s) remaining.",
  "details": {
    "current_plan_id": "SMART_199",
    "new_plan_id": "PRO_599",
    "current_fee": 199.0,
    "new_fee": 599.0,
    "proration_amount": 26.67,
    "days_remaining": 2
  }
}


## Layer 2 — Same MCP server, now driven by an LLM

What we just did by hand — pick a tool, call it, look at results, decide what to call next — is exactly what an LLM agent does in a loop. Below we wire Azure OpenAI to the same MCP server.

### The agent loop in plain English

1. Send the user's message + the tool list to the model.
2. The model either writes a tool call (`finish_reason == "tool_calls"`) or final text (`finish_reason == "stop"`).
3. If a tool call: execute it via MCP, append the result to the conversation, loop.
4. If final text: return it to the user.

That's it. ~30 lines of code below.

In [ ]:
from openai import AsyncAzureOpenAI

# Reads from .env: AZURE_OPENAI_DEPLOYMENT (with sensible default for unset case).
DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o')

# Reasoning models (o1, o3, o4-mini, etc.) take different params:
# `max_completion_tokens` instead of `max_tokens`, and they don't accept custom
# `temperature` values (only the default 1.0). Detect by deployment-name prefix.
import re
_RX = re.compile(r'(?:^|[-_/])o[1-9](?:[-_/]|$)')
IS_REASONING = bool(_RX.search(DEPLOYMENT.lower()))

client = AsyncAzureOpenAI(
    api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21'),
    # api_key + azure_endpoint come from env: AZURE_OPENAI_API_KEY + AZURE_OPENAI_ENDPOINT
)
print(f'Azure OpenAI client ready, deployment: {DEPLOYMENT}')
print(f'Reasoning model     : {IS_REASONING}')

### Translate MCP tool definitions to OpenAI shape

MCP and OpenAI both use JSON Schema for tool inputs but the envelope differs. MCP gives flat `{name, description, inputSchema}`; OpenAI wants `{type:"function", function:{name, description, parameters}}`.

In [27]:
def mcp_to_openai_tools(mcp_tools):
    return [
        {
            'type': 'function',
            'function': {
                'name': t.name,
                'description': (t.description or '').strip(),
                'parameters': t.inputSchema or {'type': 'object', 'properties': {}},
            },
        }
        for t in mcp_tools
    ]

openai_tools = mcp_to_openai_tools(tools)
print(f'Translated {len(openai_tools)} tools.')
print('\nExample (one tool in OpenAI shape):')
print(json.dumps(openai_tools[0], indent=2))

Translated 14 tools.

Example (one tool in OpenAI shape):
{
  "type": "function",
  "function": {
    "name": "get_customer_profile",
    "description": "Get a customer's profile: name, phone, email, account type (prepaid/postpaid),\n        account status (active/suspended/blocked), prepaid balance, and area code.\n        Use this when the user asks about their account or you need to identify them.",
    "parameters": {
      "properties": {
        "customer_id": {
          "title": "Customer Id",
          "type": "string"
        }
      },
      "required": [
        "customer_id"
      ],
      "title": "get_customer_profileArguments",
      "type": "object"
    }
  }
}


### The minimal agent loop

This is a stripped-down version of `src/chatbot/core/llm_orchestrator.py` — same logic, fewer features, easier to read.

In [ ]:
async def agent_loop(user_message: str, customer_id: str, max_iterations: int = 6, verbose: bool = True):
    """Run a single chat turn with tool use. Returns final text."""
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are TelcoBot, a customer support agent. '
                f'Current authenticated customer: customer_id={customer_id}. '
                f'Use this id in all tool calls; do not ask the user for it. '
                f'Greet the customer by name when you know it. '
                f'Before any mutating action, ask for confirmation. '
                f'Use ₹ for currency. Keep replies concise.'
            ),
        },
        {'role': 'user', 'content': user_message},
    ]
    if verbose:
        print(f'\n┌─ USER ({customer_id}): {user_message}')

    for i in range(max_iterations):
        # Reasoning models (o1/o3/o4-mini) and chat models use different params.
        params = {
            'model': DEPLOYMENT,
            'messages': messages,
            'tools': openai_tools,
            'tool_choice': 'auto',
        }
        if IS_REASONING:
            params['max_completion_tokens'] = 1024
        else:
            params['max_tokens'] = 1024
            params['temperature'] = 0.2

        resp = await client.chat.completions.create(**params)
        msg = resp.choices[0].message
        finish_reason = resp.choices[0].finish_reason

        record = {'role': 'assistant', 'content': msg.content}
        if msg.tool_calls:
            record['tool_calls'] = [
                {'id': tc.id, 'type': 'function',
                 'function': {'name': tc.function.name, 'arguments': tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(record)

        if finish_reason != 'tool_calls' or not msg.tool_calls:
            if verbose:
                print(f'└─ ASSISTANT: {msg.content}\n')
            return msg.content

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments) if tc.function.arguments else {}
            if verbose:
                print(f'│  → tool: {tc.function.name}({args})')
            result = await call_mcp_tool(tc.function.name, args)
            preview = result['content'][:140].replace('\n', ' ')
            if verbose:
                marker = '✗' if result['is_error'] else '✓'
                print(f'│    {marker} {preview}{"..." if len(result["content"]) > 140 else ""}')
            messages.append({
                'role': 'tool',
                'tool_call_id': tc.id,
                'content': result['content'] or '(empty)',
            })

    if verbose:
        print('└─ (hit iteration cap)')
    return None

print('Agent loop defined.')

### Demo 1 — single tool call

Simple question. The model should use one tool and answer.

In [29]:
_ = await agent_loop('what plan am I on?', 'CUST002')


┌─ USER (CUST002): what plan am I on?


BadRequestError: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param': 'max_tokens', 'code': 'unsupported_parameter'}}

### Demo 2 — multi-tool reasoning

Open-ended question. The model should chain tools: check usage, find that data is nearly exhausted, look up data add-ons, then propose options.

In [ ]:
_ = await agent_loop('my internet feels slow today', 'CUST002')

### Demo 3 — diagnostic with a different customer

CUST003 has a suspended account because of an overdue bill. The model has to figure out *why* the number isn't working.

In [ ]:
_ = await agent_loop('why is my number not working?', 'CUST003')

### Demo 4 — network outage

CUST005 is in area BLR-04, where the seed data has an active fiber-cut outage.

In [ ]:
_ = await agent_loop('calls keep dropping in the evening, whats going on?', 'CUST005')

## Takeaways

- **Layer 1** showed that the MCP server is just a normal HTTP service speaking a standard protocol. Anything that speaks MCP can use these tools — Claude Desktop, Cursor, your own agent, etc.
- **Layer 2** showed that an LLM agent is the same MCP calls, but orchestrated in a loop where the *model* decides which tool to invoke next based on the running conversation.
- The MCP server has **no idea an LLM is involved**. It just answers `list_tools` and `call_tool` requests. That's the whole point of the protocol — decouple the tool surface from the consumer.
- The minimal agent loop here is ~30 lines. The full version in `src/chatbot/core/llm_orchestrator.py` adds: prompt-caching-friendly system prompt placement, structured turn logging, error wrapping, multi-skill routing slots. The core logic is identical.

## To extend

- Add a new tool: edit `src/mcp_servers/telecom/tools.py` (a few lines), restart `mcp_telecom`, re-run cell 4 (list tools) to see it appear, re-run any agent demo — the model will pick it up automatically.
- Replace the LLM: swap the `AsyncAzureOpenAI` instantiation. The MCP layer doesn't change.
- Replace the tools entirely (banking, healthcare, e-commerce): write a new MCP server, point the chatbot at it. Same loop, different domain.